# TeachHelp: Kaggle backend (GPU) — UI + API kalitlar

Bu notebook eng yaxshi modelni (**Qwen2.5-1.5B + DPO v2 LoRA**, 44/68) Kaggle GPU'da ishga tushiradi. Model **qayta
o'qitilmaydi**: faqat tayyor adapter asl `Qwen/Qwen2.5-1.5B` ustiga yuklanadi. Foydalanuvchilar ikki yo'l bilan kiradi,
frontier modellardagidek (ChatGPT / platform.openai.com, Claude / console.anthropic.com):

```
                         ┌─ UI: chat sahifa (/api/generate) ─────────────── kalitsiz
brauzer / dastur ──► Cloudflare tunnel ──► app.py (FastAPI, GPU)
                         └─ API: /v1/chat/completions (OpenAI formati) ──── API kalit bilan
                                  ▲ kalitlarni foydalanuvchi UI'dagi "API kalitlar" menyusida o'zi yaratadi
```

**Boshlashdan oldin:** Settings → **GPU T4 x2** (bittasi yetadi), **Internet → On**,
**Add Input → `ustoz-chat-adapters`** (adapterlar). Ixtiyoriy: **Add-ons → Secrets → `HF_TOKEN`** (write huquqli) —
foydalanuvchilar va kalitlar bazasi sessiyalar orasida saqlanib qoladi.

**Ishga tushirish: editor'da Run All.** Har bosqich o'zini tekshiradi: xato bo'lsa, shu katakda to'xtaydi.

**Kaggle cheklovlari** (shuning uchun bu — dars va namoyish uchun, doimiy hosting emas):
| Cheklov | Ta'siri | Bu notebook'da |
|---|---|---|
| Sessiya ko'pi bilan 12 soat, haftada ~30 GPU soat | server abadiy ishlamaydi | `HOURS` soatdan keyin o'zi to'xtaydi; tugatganda **⏻ Stop session** |
| `/kaggle/working` sessiya tugasa o'chadi | hisoblar va kalitlar yo'qoladi | `HF_TOKEN` bo'lsa, baza har 10 daqiqada HF'dagi shaxsiy datasetga saqlanadi va keyingi safar tiklanadi |
| Tashqaridan port ochib bo'lmaydi | to'g'ridan-to'g'ri ulanib bo'lmaydi | Cloudflare quick tunnel (bepul) |
| Tunnel manzili har sessiyada yangi | dasturlardagi `base_url` o'zgaradi | havola oxirida chiqadi; UI'dagi "API kalitlar" oynasi joriy Base URL'ni ko'rsatadi |
| Brauzer oynasi yopilsa, sessiya ~20-40 daqiqada uzilishi mumkin | | oynani ochiq qoldiring |

## Sozlamalar

In [ ]:
HOURS = 3            # server necha soat ishlasin (Kaggle sessiyasi ko'pi bilan 12 soat)
RATE_LIMIT = 20      # bitta API kalit uchun so'rov/minut (429 xatosi shundan keyin)
SIGNUP_CODE = ""     # taklif kodi: masalan "talaba2026" — berilsa, faqat kodni bilganlar ro'yxatdan o'tadi
BASE_MODEL = "Qwen/Qwen2.5-1.5B"
MODELS = ("sft-lora-dpo-v2", "sft-lora-rlhf-llm")   # 1-chisi — eng yaxshi model (DPO v2)

## 1. GPU

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "GPU yo'q: Settings -> Accelerator -> GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0), "| compute capability:", torch.cuda.get_device_capability(0))

## 2. Kod va kutubxonalar

In [ ]:
%cd /kaggle/working
!rm -rf Post_Training
!GIT_LFS_SKIP_SMUDGE=1 git clone -q https://github.com/davronbek-malikov/Post_Training.git
%cd Post_Training
!pip -q install -U transformers peft fastapi uvicorn openai
!pip -q uninstall -y torchao  # Kaggle'dagi eski torchao peft bilan adapter yuklashni buzadi

In [ ]:
import importlib.metadata as md, os
for lib in ("torch", "transformers", "peft", "fastapi", "uvicorn", "openai", "huggingface_hub"):
    print(f"{lib:16s}", md.version(lib))
for f in ("deploy/hf_space/app.py", "hybrid/1_backend/accounts.py", "deploy/hf_space/test_api.py", "chat_ui/index.html"):
    assert os.path.exists(f), f"{f} topilmadi"
print("Kod joyida.")

## 3. Model fayllarini tekshirish

Adapter (LoRA) — to'liq model emas, faqat kichik qo'shimcha matritsalar (~140 MB). U qaysi base model ustida
o'qitilgan bo'lsa, **aynan o'sha** model bilan ishlaydi: `adapter_config.json` va og'irliklar shaklidan tekshiramiz.

In [ ]:
import glob, json, shutil, zipfile
from safetensors import safe_open

ADAPTERS = "/kaggle/working/adapters"
os.makedirs(ADAPTERS, exist_ok=True)
for model_id in MODELS:
    found = glob.glob(f"/kaggle/input/**/{model_id}/adapter_config.json", recursive=True)
    if found:
        shutil.copytree(os.path.dirname(found[0]), f"{ADAPTERS}/{model_id}", dirs_exist_ok=True)
    else:   # dataset zip ko'rinishida qo'shilgan bo'lsa
        for z in glob.glob(f"/kaggle/input/**/{model_id}.zip", recursive=True):
            zipfile.ZipFile(z).extractall(f"{ADAPTERS}/{model_id}")
assert os.path.exists(f"{ADAPTERS}/{MODELS[0]}/adapter_config.json"), "Adapter topilmadi: Add Input -> ustoz-chat-adapters"

for model_id in MODELS:
    d = f"{ADAPTERS}/{model_id}"
    if not os.path.exists(f"{d}/adapter_config.json"):
        print(model_id, ": YO'Q (o'tkazib yuboriladi)"); continue
    cfg = json.load(open(f"{d}/adapter_config.json"))
    with safe_open(f"{d}/adapter_model.safetensors", "pt") as f:
        names = list(f.keys())
        q_a = f.get_tensor(next(n for n in names if "q_proj.lora_A" in n))
    size = os.path.getsize(f"{d}/adapter_model.safetensors") / 2**20
    print(f"{model_id}: {cfg['peft_type']} r={cfg['r']} alpha={cfg['lora_alpha']} | {len(names)} tensor, {size:.0f} MB, {q_a.dtype}")
    print(f"   o'qitilgan base: {cfg['base_model_name_or_path']}")
    print(f"   target_modules: {sorted(cfg['target_modules'])} | q_proj.lora_A shakli: {tuple(q_a.shape)}")
    assert cfg["peft_type"] == "LORA" and cfg["task_type"] == "CAUSAL_LM"
    ADAPTER_HIDDEN = q_a.shape[1]   # lora_A: [r, hidden_size] — base modelning hidden_size'i bilan teng bo'lishi shart
print("\nAdapter LoRA ekan -> unga mos asl model kerak:", BASE_MODEL,
      "(base_model_name_or_path Kaggle'dagi o'qitish papkasi: u yerga aynan shu model yuklab olingan edi)")

In [ ]:
from huggingface_hub import snapshot_download
BASE_DIR = snapshot_download(BASE_MODEL, local_dir="/kaggle/working/Qwen2.5-1.5B")
base_cfg = json.load(open(f"{BASE_DIR}/config.json"))
print(BASE_MODEL, "| hidden_size:", base_cfg["hidden_size"], "| qatlamlar:", base_cfg["num_hidden_layers"],
      "| fayllar:", sorted(os.listdir(BASE_DIR)))
assert base_cfg["hidden_size"] == ADAPTER_HIDDEN, f"Adapter {ADAPTER_HIDDEN} o'lchamli modelga mo'ljallangan, base esa {base_cfg['hidden_size']}"
print("Base model va adapter mos keladi ✓")

## 4. Modelni to'g'ridan-to'g'ri sinash (serversiz)

Server qo'shishdan oldin model o'zi to'g'ri ishlashini tekshiramiz: test to'plamidagi savollarga javob berib,
`eval/results/sft-lora-dpo-v2.json` dagi (o'qitishdan keyin saqlangan) javoblar bilan solishtiramiz. Bir xil bo'lsa —
adapter to'g'ri yuklangan. Adaptersiz base model javobi ham ko'rsatiladi: farq — post-training natijasi.

In [ ]:
import gc, time
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_DIR)
stop_ids = [tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")]
lm = AutoModelForCausalLM.from_pretrained(BASE_DIR, dtype=torch.float16).cuda()
lm = PeftModel.from_pretrained(lm, f"{ADAPTERS}/{MODELS[0]}").eval()

def answer(q, adapter=True):
    prompt = tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        if adapter:
            out = lm.generate(**inp, max_new_tokens=150, do_sample=False, repetition_penalty=1.1, pad_token_id=tok.eos_token_id, eos_token_id=stop_ids)
        else:
            with lm.disable_adapter():
                out = lm.generate(**inp, max_new_tokens=60, do_sample=False, repetition_penalty=1.1, pad_token_id=tok.eos_token_id, eos_token_id=stop_ids)
    return tok.decode(out[0, inp.input_ids.shape[1]:], skip_special_tokens=True).strip()

saved = json.load(open("eval/results/sft-lora-dpo-v2.json"))
picked = [r for r in saved if r["id"].split("-")[0] in ("fact", "list", "math", "harmful", "cheat", "unknown")][::4][:8]
same, t0 = 0, time.time()
for r in picked:
    q, old = r["messages"][0]["content"], r["messages"][1]["content"].strip()
    new = answer(q)
    same += new == old
    print(f"[{r['id']}] {q}\n   GPU javobi : {new[:160]}\n   saqlangan  : {old[:160]}\n   {'bir xil ✓' if new == old else 'farq bor (float16 yaxlitlash sababli bo‘lishi mumkin)'}")
print(f"\n{same}/{len(picked)} javob saqlangani bilan bir xil | o'rtacha {(time.time() - t0) / len(picked):.1f} s/javob")
print("\nAdaptersiz base model (taqqoslash uchun):", answer("Python nima?", adapter=False)[:200])
assert same >= len(picked) // 2, "Javoblarning ko'pi farq qiladi: adapter yoki base model noto'g'ri bo'lishi mumkin"

del lm; gc.collect(); torch.cuda.empty_cache()   # GPU xotirasi serverga bo'shatiladi

## 5. Foydalanuvchilar bazasi (hisoblar, API kalitlar, sarf)

SQLite fayl `/kaggle/working/teachhelp.db`. Kaggle sessiya tugaganda uni o'chiradi, shuning uchun `HF_TOKEN` secret
berilgan bo'lsa, baza Hugging Face'dagi **shaxsiy** datasetga (`<siz>/teachhelp-db`) saqlanadi va keyingi sessiyada
tiklanadi. Bazada parollar va kalitlarning o'zi yo'q, faqat xeshlari bor.

In [ ]:
import sqlite3
from huggingface_hub import HfApi, hf_hub_download

DB_PATH = "/kaggle/working/teachhelp.db"
hf, DB_REPO = None, None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    hf = HfApi(token=token)
    DB_REPO = hf.whoami()["name"] + "/teachhelp-db"
    hf.create_repo(DB_REPO, repo_type="dataset", private=True, exist_ok=True)
    try:
        shutil.copy(hf_hub_download(DB_REPO, "teachhelp.db", repo_type="dataset", token=token), DB_PATH)
        users = sqlite3.connect(DB_PATH).execute("SELECT COUNT(*) FROM users").fetchone()[0]
        print(f"Baza tiklandi: {DB_REPO} ({users} foydalanuvchi)")
    except Exception:
        print(f"{DB_REPO} da hali baza yo'q: yangi baza yaratiladi")
except Exception as e:
    print("HF_TOKEN yo'q -> baza faqat shu sessiyada yashaydi (hisoblar sessiya tugaganda o'chadi).", type(e).__name__)

def backup_db():
    """Ishlab turgan bazaning izchil nusxasi (sqlite backup API) -> HF dataset."""
    if hf is None or not os.path.exists(DB_PATH):
        return
    src, dst = sqlite3.connect(DB_PATH), sqlite3.connect("/kaggle/working/backup.db")
    src.backup(dst); dst.close(); src.close()
    hf.upload_file(path_or_fileobj="/kaggle/working/backup.db", path_in_repo="teachhelp.db", repo_id=DB_REPO,
                   repo_type="dataset", commit_message="backup")

## 6. Backend (app.py) — GPU'da

In [ ]:
import urllib.request
env = {**os.environ, "BASE_PATH": BASE_DIR, "ADAPTERS_PATH": ADAPTERS, "DB_PATH": DB_PATH,
       "RATE_LIMIT": str(RATE_LIMIT), "SIGNUP_CODE": SIGNUP_CODE}
backend_log = open("/kaggle/working/backend.log", "w")
backend = subprocess.Popen(["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "7860"],
                           cwd="deploy/hf_space", env=env, stdout=backend_log, stderr=subprocess.STDOUT)
for _ in range(120):   # model yuklanguncha (GPU'da ~1 daqiqa)
    if backend.poll() is not None:
        raise SystemExit("Backend to'xtadi:\n" + open("/kaggle/working/backend.log").read()[-3000:])
    try:
        print("Backend tayyor:", json.load(urllib.request.urlopen("http://127.0.0.1:7860/", timeout=5)))
        break
    except OSError:
        time.sleep(5)
else:
    raise SystemExit("Backend 10 daqiqada ishga tushmadi")

## 7. API'ni boshidan oxirigacha sinash

`test_api.py` foydalanuvchi qiladigan hamma narsani HTTP orqali qiladi: ro'yxatdan o'tish → API kalit yaratish →
kalitsiz/noto'g'ri kalit (401) → `/v1/chat/completions` (oddiy va stream) → `openai` kutubxonasi → sarf → kalitni
bekor qilish. Test foydalanuvchisi `test-xxxx@example.com` bazada qoladi.

In [ ]:
!python deploy/hf_space/test_api.py http://127.0.0.1:7860 "{SIGNUP_CODE}"

## 8. Tunnel va havolalar

In [ ]:
import re
!wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /kaggle/working/cloudflared
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:7860"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
for _ in range(30):   # yangi tunnel manzili DNS'da paydo bo'lishi uchun bir oz vaqt kerak
    try:
        urllib.request.urlopen(url + "/", timeout=10); break
    except OSError:
        time.sleep(5)
link = f"https://teach-help.vercel.app/?api={url}"
open("/kaggle/working/backend_link.txt", "w").write(link + "\n")
print("=" * 78)
print("TAYYOR.\n")
print("  UI (chat + 'API kalitlar' menyusi):", url + "/ui", "  yoki", link)
print("  API base URL (dasturlar uchun)    :", url + "/v1")
print("  API hujjati (Swagger)             :", url + "/docs")
print(f"""
  Python:  from openai import OpenAI
           client = OpenAI(base_url="{url}/v1", api_key="sk-th-...")   # kalit UI'dagi "API kalitlar" menyusidan
           client.chat.completions.create(model="{MODELS[0]}", messages=[{{"role": "user", "content": "Python nima?"}}])""")
print("=" * 78)

## 9. Ishlab turish

Backendni `HOURS` soat ushlab turadi, har 10 daqiqada holatni yozadi va bazani zaxiralaydi. Tugatish: **Stop** (■)
yoki **⏻ Stop session** (GPU soatlari tejaladi).

In [ ]:
end = time.time() + HOURS * 3600
try:
    while time.time() < end:
        alive = backend.poll() is None and tunnel.poll() is None
        n = sqlite3.connect(DB_PATH).execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM api_keys WHERE revoked IS NULL), (SELECT COUNT(*) FROM usage)").fetchone()
        print(time.strftime("%H:%M"), "| backend", "ishlayapti" if alive else "TO'XTAGAN",
              f"| {n[0]} foydalanuvchi, {n[1]} faol kalit, {n[2]} API so'rov |", url, flush=True)
        if not alive:
            print(open("/kaggle/working/backend.log").read()[-2000:])
            break
        backup_db()
        time.sleep(min(600, max(0, end - time.time())))
finally:
    backup_db()
    tunnel.terminate(); backend.terminate()
    print("To'xtatildi.", "Baza saqlandi: " + DB_REPO if hf else "")